In [1]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

c:\Users\CHITTA\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

In [3]:
import os
os.makedirs("data", exist_ok=True)

for i, doc in enumerate(sample_docs):
    with open(f"data/doc_{i}.txt", "w") as f:
        f.write(doc)

In [4]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[2].page_content[:300] + "...")

C:\Users\CHITTA\AppData\Local\Temp\ipykernel_22904\1388857567.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Loaded 3 documents

First document preview:

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily...


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,     # Maximum size of each chunk
    chunk_overlap = 50,   # Overlap between chunks to maintain context
    length_function = len,
    separators = [" ", "\n\n", "\n", "."]    # Hierarchy of separators
)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Metadata: {'source': 'data\\doc_0.txt'}


In [6]:
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = persist_directory,
    collection_name = "rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 5 vectors
Persisted to: ./chroma_db


In [10]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query, k = 3)
for doc in similar_docs:
    print(doc.page_content)
    print("-" * 50)

Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through
--------------------------------------------------
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
--------------------------------------------------
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech re

In [11]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What are the types of machine learning?

Top 3 similar chunks:

--- Chunk 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 2 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....
Source: data\doc_1.txt

--- Chunk 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt


In [12]:
results = vectorstore.similarity_search_with_score(query, k = 3)
for doc, score in results:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}")
    print("-" * 50)

Score: 0.551533043384552
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through
--------------------------------------------------
Score: 1.026219367980957
Content: Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
--------------------------------------------------
Score: 1.0638008117675781
Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep l

## Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="gemma3:latest",
    temperature=0.7,
)

In [4]:
test_response = llm.invoke("What is machine learning?")
print(test_response.content)

Okay, let's break down what machine learning (ML) is. It’s a really fascinating and rapidly growing field! Here's a breakdown in layers:

**1. The Basic Idea:**

At its core, machine learning is about teaching computers to *learn* from data without being explicitly programmed for every single scenario. Instead of giving a computer precise instructions on how to solve a problem (like "if X then do Y"), you feed it lots of data and let it figure out the patterns itself. 

**2. How It Works – The Key Steps:**

* **Data Collection:**  You start with a dataset - this could be anything: images, text, numbers, sensor readings, etc. The more relevant and high-quality data you have, generally the better your machine learning model will perform.
* **Feature Extraction (Sometimes):** This involves identifying and selecting the most important characteristics (features) within the data that are useful for making predictions.  For example, if you're trying to predict house prices, features might inc